[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.6 MB/s eta 0:00:00


In [2]:
import torch
import math

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    B, S, D = q.shape
    i = torch.arange(D//2) # we need pairs
    inv_freq = 1.0 / (10000 ** (2 * i.float() / D))
    pos = torch.arange(S)
    pos = pos.view(S, 1)

    theta = pos * inv_freq

    # 1. Compute position angles
    # 2. Split into even/odd pairs
    q0 = q[..., 0::2]
    q1 = q[..., 1::2]

    k0 = k[..., 0::2]
    k1 = k[..., 1::2]


    # 3. Apply rotation
    q_rot0 = q0*torch.cos(theta) - q1*torch.sin(theta)
    q_rot1 = q0*torch.sin(theta) + q1*torch.cos(theta)

    k_rot0 = k0*torch.cos(theta) - k1*torch.sin(theta)
    k_rot1 = k0*torch.sin(theta) + k1*torch.cos(theta)

    qr = torch.stack([q_rot0, q_rot1], dim=-1)
    kr = torch.stack([k_rot0, k_rot1], dim=-1)

    qr = qr.view(B, S, D)
    kr = kr.view(B, S, D)

    return (qr, kr)

In [7]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (1.8ms)
  ✅ [2/4] Preserves norm (2.2ms)
  ✅ [3/4] Relative position property (1.7ms)
  ✅ [4/4] Gradient flow (0.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (6.5ms total)
  Progress saved. Run status() to see your dashboard.

